# Neural Network Models for Cascading Disaster Prediction

**Four architectures with Optuna hyperparameter tuning:**

| # | Model | Description |
|---|-------|-------------|
| 1 | **Baseline MLP** | 3-layer feed-forward network with full Optuna HP search |
| 2 | **MLP + Weather Embedding** | Same MLP backbone + a physics-informed weather embedding that encodes meteorological priors |
| 3 | **Autoencoder + Classifier** | Learns a compressed representation via reconstruction, then classifies from the bottleneck |
| 4 | **Autoencoder + Weather Embedding** | Autoencoder with weather-aware bottleneck + physics-informed embedding |

### Weather Embedding — Physics-Informed Design
The weather embedding is a **trainable module** that encodes domain knowledge:
- Separates raw features into **meteorological groups** (temporal/cyclical, spatial/geographic, severity/impact, event-type indicators, cascade-transition probabilities, historical context)
- Each group passes through its own sub-network with **physics-inspired activation constraints** (e.g., cyclical features use sine/cosine gating; spatial features use radial-basis-function-like transforms)
- Group embeddings are fused via a learned **attention-weighted combination**, giving the model an inductive bias toward physically meaningful feature interactions
- The fused embedding is concatenated with the raw features before the classifier head

In [11]:
import numpy as np
import pandas as pd
import pickle
import time
import warnings
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, hamming_loss,
    precision_recall_curve, confusion_matrix, roc_curve, auc
)

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}  •  Device: {DEVICE}")

%matplotlib inline

PyTorch 2.10.0+cu128  •  Device: cuda


In [9]:
!pip install xgboost optuna scikit-learn matplotlib seaborn torch torchvision torchaudio --quiet

In [ ]:
# ── Feature Toggle ──────────────────────────────────────────────────
# Set to True to EXCLUDE p(secondary | primary) features and
# total_cascade_probability.  Matches the XGBoost notebook toggle.
EXCLUDE_CASCADE_PROB_FEATURES = True

In [ ]:
DATA_DIR = Path('../data/chronological_prepared_data')

X_train_raw = np.load(DATA_DIR / 'X_train.npy')
X_test_raw  = np.load(DATA_DIR / 'X_test.npy')
y_train_raw = np.load(DATA_DIR / 'y_train.npy')
y_test_raw  = np.load(DATA_DIR / 'y_test.npy')

with open(DATA_DIR / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)

feature_names = list(metadata['feature_names'])
target_names  = metadata['target_names']

print(f"Train : {X_train_raw.shape[0]:,} samples, {X_train_raw.shape[1]} features")
print(f"Test  : {X_test_raw.shape[0]:,} samples")
print(f"Labels: {len(target_names)}")
print(f"Split : {metadata.get('split_type', 'unknown')}")

# ── Optionally remove cascade-probability features ──────────────────
if EXCLUDE_CASCADE_PROB_FEATURES:
    keep_mask = []
    removed = []
    for i, name in enumerate(feature_names):
        if (name.startswith('p_') and name.endswith('_given_primary')) or name == 'total_cascade_probability':
            removed.append(name)
            keep_mask.append(False)
        else:
            keep_mask.append(True)
    keep_mask = np.array(keep_mask)
    X_train_raw = X_train_raw[:, keep_mask]
    X_test_raw  = X_test_raw[:, keep_mask]
    feature_names = [n for n, k in zip(feature_names, keep_mask) if k]
    print(f"\n⚠ Excluded {len(removed)} cascade-probability features:")
    for r in removed:
        print(f"  - {r}")
    print(f"Remaining features: {len(feature_names)}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/chronological_prepared_data/X_train.npy'

In [ ]:
# ── Build feature-group indices for the Weather Embedding ──────────────
# The physics-informed embedding needs to know which columns belong to which
# meteorological concept so it can apply appropriate inductive biases.

def build_feature_groups(feature_names):
    """Partition feature indices into semantically meaningful groups."""
    groups = {
        'temporal_cyclical': [],   # sin/cos encoded time features
        'temporal_ordinal':  [],   # day_of_year, hour, etc.
        'spatial':           [],   # lat, lon, coastal, coordinate features
        'severity_impact':   [],   # damage, injuries, deaths, severity scores
        'event_type':        [],   # one-hot event-type indicators + group flags
        'cascade_prob':      [],   # p_*_given_primary transition probabilities
        'historical':        [],   # rolling windows, days_since, cascade counts
        'other':             [],   # everything else
    }
    
    cyclical_kw = ['_sin', '_cos']
    temporal_kw = ['day_of_year', 'day_of_week', 'month', 'hour', 'event_duration']
    spatial_kw  = ['lat', 'lon', 'coastal', 'coordinate', 'BEGIN_LAT', 'BEGIN_LON',
                   'LATITUDE', 'LONGITUDE', 'abs_latitude', 'begin_range',
                   'event_path_length', 'is_marine', 'is_zone']
    severity_kw = ['damage', 'injur', 'death', 'fatal', 'severity', 'magnitude',
                   'log_damage', 'has_magnitude', 'is_severe_wind', 'is_significant_hail']
    etype_kw    = ['etype_', 'is_hurricane', 'is_flood_type', 'is_winter_type',
                   'is_convective', 'event_rarity']
    cascade_kw  = ['p_', '_given_primary', 'total_cascade_prob']
    hist_kw     = ['events_last_', 'damage_last_', 'max_severity_last_', 'cascades_',
                   'days_since_', 'recent_event_density', 'location_event_count',
                   'location_avg', 'location_cascade', 'state_avg', 'state_cascade',
                   'event_type_avg']
    
    for i, name in enumerate(feature_names):
        nl = name.lower()
        if any(k in nl for k in cyclical_kw):
            groups['temporal_cyclical'].append(i)
        elif any(k in nl for k in temporal_kw):
            groups['temporal_ordinal'].append(i)
        elif name.startswith('p_') and 'given_primary' in nl:
            groups['cascade_prob'].append(i)
        elif any(k in nl for k in spatial_kw):
            groups['spatial'].append(i)
        elif any(k in nl for k in severity_kw):
            groups['severity_impact'].append(i)
        elif any(k in nl for k in etype_kw):
            groups['event_type'].append(i)
        elif any(k in nl for k in hist_kw):
            groups['historical'].append(i)
        else:
            groups['other'].append(i)
    
    # Merge tiny groups into 'other'
    for k in list(groups.keys()):
        if len(groups[k]) == 0 and k != 'other':
            del groups[k]
    
    return groups

FEATURE_GROUPS = build_feature_groups(feature_names)
print("Feature groups:")
for g, idx_list in FEATURE_GROUPS.items():
    print(f"  {g:25s} : {len(idx_list):3d} features")
print(f"  {'TOTAL':25s} : {sum(len(v) for v in FEATURE_GROUPS.values()):3d}")

Feature groups:
  temporal_cyclical         :   6 features
  temporal_ordinal          :   5 features
  spatial                   :  16 features
  severity_impact           :  31 features
  event_type                :  59 features
  cascade_prob              :  29 features
  historical                :   6 features
  other                     :  13 features
  TOTAL                     : 165


In [ ]:
# ── PyTorch data loaders ────────────────────────────────────────────────
def make_loaders(X_tr, y_tr, X_val, y_val, X_te, y_te, batch_size=1024):
    train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                             torch.tensor(y_tr, dtype=torch.float32))
    val_ds   = TensorDataset(torch.tensor(X_val, dtype=torch.float32),
                             torch.tensor(y_val, dtype=torch.float32))
    test_ds  = TensorDataset(torch.tensor(X_te, dtype=torch.float32),
                             torch.tensor(y_te, dtype=torch.float32))
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader, test_loader

print("Data loader factory ready.")

Data loader factory ready.


In [ ]:
# ── Training loop (shared by all models) ───────────────────────────────
def train_model(model, train_loader, val_loader, pos_weight_tensor,
                lr=1e-3, weight_decay=1e-4, epochs=80, patience=12,
                verbose=True, recon_weight=0.0):
    """Train a model with BCE loss, optional reconstruction loss, early stopping."""
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
    recon_criterion = nn.MSELoss()
    
    best_val_f1 = -1
    best_state = None
    wait = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    
    for epoch in range(1, epochs + 1):
        # ── Train ──
        model.train()
        epoch_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            
            out = model(xb)
            if isinstance(out, tuple):  # (logits, reconstruction)
                logits, recon = out
                loss = criterion(logits, yb) + recon_weight * recon_criterion(recon, xb)
            else:
                logits = out
                loss = criterion(logits, yb)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item() * xb.size(0)
        epoch_loss /= len(train_loader.dataset)
        
        # ── Validate ──
        model.eval()
        val_loss = 0
        all_logits, all_y = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb)
                if isinstance(out, tuple):
                    logits, recon = out
                    loss = criterion(logits, yb) + recon_weight * recon_criterion(recon, xb)
                else:
                    logits = out
                    loss = criterion(logits, yb)
                val_loss += loss.item() * xb.size(0)
                all_logits.append(logits.cpu())
                all_y.append(yb.cpu())
        val_loss /= len(val_loader.dataset)
        
        all_logits = torch.cat(all_logits).numpy()
        all_y = torch.cat(all_y).numpy()
        preds = (torch.sigmoid(torch.tensor(all_logits)).numpy() >= 0.5).astype(int)
        val_f1 = f1_score(all_y, preds, average='macro', zero_division=0)
        
        scheduler.step(val_f1)
        history['train_loss'].append(epoch_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        
        if verbose and (epoch % 10 == 0 or epoch == 1 or wait == 0):
            marker = ' ★' if wait == 0 else ''
            print(f"  Epoch {epoch:3d}  loss={epoch_loss:.4f}  val_loss={val_loss:.4f}  "
                  f"val_f1={val_f1:.4f}{marker}")
        
        if wait >= patience:
            if verbose:
                print(f"  Early stop at epoch {epoch} (best val_f1={best_val_f1:.4f})")
            break
    
    model.load_state_dict(best_state)
    model.to(DEVICE)
    return model, history, best_val_f1


# ── Prediction & threshold tuning ──────────────────────────────────────
def predict_proba(model, loader):
    """Return sigmoid probabilities."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(DEVICE)
            out = model(xb)
            logits = out[0] if isinstance(out, tuple) else out
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def tune_thresholds(y_true, y_prob, min_thresh=0.05):
    """Per-label threshold tuning on validation set (maximise F1)."""
    n_labels = y_true.shape[1]
    thresholds = np.full(n_labels, 0.5)
    for i in range(n_labels):
        if y_true[:, i].sum() == 0:
            continue
        precs, recs, ths = precision_recall_curve(y_true[:, i], y_prob[:, i])
        f1s = 2 * precs * recs / (precs + recs + 1e-8)
        best = np.argmax(f1s)
        if best < len(ths):
            thresholds[i] = max(ths[best], min_thresh)
    return thresholds


def evaluate_model(model_name, y_true, y_prob, thresholds, target_names):
    """Compute and return a results dict + per-label DataFrame."""
    y_pred = (y_prob >= thresholds).astype(int)
    overall = {
        'model': model_name,
        'f1_weighted':     f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_macro':        f1_score(y_true, y_pred, average='macro',    zero_division=0),
        'f1_micro':        f1_score(y_true, y_pred, average='micro',    zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro':    recall_score(y_true, y_pred, average='macro',    zero_division=0),
        'hamming_loss':    hamming_loss(y_true, y_pred),
        'subset_accuracy': (y_true == y_pred).all(axis=1).mean(),
    }
    per_label = []
    for j, lbl in enumerate(target_names):
        sup = int(y_true[:, j].sum())
        f1   = f1_score(y_true[:, j], y_pred[:, j], zero_division=0)
        prec = precision_score(y_true[:, j], y_pred[:, j], zero_division=0)
        rec  = recall_score(y_true[:, j], y_pred[:, j], zero_division=0)
        try:
            prauc = average_precision_score(y_true[:, j], y_prob[:, j])
        except Exception:
            prauc = 0.0
        per_label.append(dict(label=lbl, support=sup, f1=f1, precision=prec, recall=rec, pr_auc=prauc))
    return overall, pd.DataFrame(per_label)


print("Training & evaluation utilities loaded.")

Training & evaluation utilities loaded.


---
## 3. Model Definitions

### 3a. Physics-Informed Weather Embedding

This reusable module encodes domain knowledge about meteorological features:
- **Group-specific sub-networks** with tailored activations
- **Cyclical gating** for temporal features (preserves periodicity)
- **RBF-like spatial transform** (latitude/longitude → distance-based representation)
- **Learned attention** over groups for fusion

In [ ]:
class CyclicalGate(nn.Module):
    """Gating layer that preserves cyclical structure: output = x * sigmoid(Wx+b) + sin(Vx+c)."""
    def __init__(self, dim):
        super().__init__()
        self.gate_linear = nn.Linear(dim, dim)
        self.wave_linear = nn.Linear(dim, dim)

    def forward(self, x):
        return x * torch.sigmoid(self.gate_linear(x)) + torch.sin(self.wave_linear(x))


class RBFSpatialTransform(nn.Module):
    """Radial-basis-function inspired layer for spatial coordinates.
    Learns K centres and widths; outputs exp(-||x - c_k||^2 / (2*sigma_k^2))."""
    def __init__(self, in_dim, n_centres=16):
        super().__init__()
        self.centres = nn.Parameter(torch.randn(n_centres, in_dim) * 0.5)
        self.log_sigma = nn.Parameter(torch.zeros(n_centres))
    
    def forward(self, x):
        # x: (B, in_dim), centres: (K, in_dim)
        diff = x.unsqueeze(1) - self.centres.unsqueeze(0)  # (B, K, in_dim)
        sq_dist = (diff ** 2).sum(dim=-1)  # (B, K)
        sigma2 = (2 * self.log_sigma.exp() ** 2).clamp(min=1e-4)
        return torch.exp(-sq_dist / sigma2)  # (B, K)


class PhysicsInformedWeatherEmbedding(nn.Module):
    """Encode raw features into a physics-aware embedding.
    
    Each feature group passes through a specialised sub-network:
    - temporal_cyclical → CyclicalGate
    - spatial           → RBFSpatialTransform + linear
    - cascade_prob      → softmax-normalised before linear (probability simplex)
    - others            → standard linear+GELU blocks
    
    Group embeddings are fused via learned attention weights.
    """
    def __init__(self, feature_groups, group_embed_dim=32, n_rbf_centres=16):
        super().__init__()
        self.feature_groups = feature_groups
        self.group_names = sorted(feature_groups.keys())
        self.group_embed_dim = group_embed_dim
        
        self.sub_networks = nn.ModuleDict()
        for gname in self.group_names:
            in_dim = len(feature_groups[gname])
            if in_dim == 0:
                continue
            if gname == 'temporal_cyclical':
                self.sub_networks[gname] = nn.Sequential(
                    nn.Linear(in_dim, group_embed_dim),
                    CyclicalGate(group_embed_dim),
                    nn.Linear(group_embed_dim, group_embed_dim),
                )
            elif gname == 'spatial':
                self.sub_networks[gname] = nn.Sequential(
                    RBFSpatialTransform(in_dim, n_centres=n_rbf_centres),
                    nn.Linear(n_rbf_centres, group_embed_dim),
                    nn.GELU(),
                )
            elif gname == 'cascade_prob':
                # Treat cascade transition probabilities as a probability vector
                self.sub_networks[gname] = nn.Sequential(
                    nn.Softmax(dim=-1),
                    nn.Linear(in_dim, group_embed_dim),
                    nn.GELU(),
                    nn.Linear(group_embed_dim, group_embed_dim),
                )
            else:  # severity_impact, event_type, historical, other, temporal_ordinal
                self.sub_networks[gname] = nn.Sequential(
                    nn.Linear(in_dim, group_embed_dim),
                    nn.GELU(),
                    nn.Linear(group_embed_dim, group_embed_dim),
                )
        
        n_groups = len(self.sub_networks)
        # Attention over group embeddings
        self.attn_query = nn.Linear(group_embed_dim, 1, bias=False)
        self.fuse = nn.Linear(n_groups * group_embed_dim, n_groups * group_embed_dim)
        self.out_dim = n_groups * group_embed_dim
    
    def forward(self, x):
        group_embeds = []
        for gname in self.group_names:
            if gname not in self.sub_networks:
                continue
            idx = self.feature_groups[gname]
            x_group = x[:, idx]
            group_embeds.append(self.sub_networks[gname](x_group))
        
        # Stack: (B, n_groups, embed_dim)
        stacked = torch.stack(group_embeds, dim=1)
        # Attention scores
        attn_scores = self.attn_query(stacked).squeeze(-1)  # (B, n_groups)
        attn_weights = F.softmax(attn_scores, dim=-1).unsqueeze(-1)  # (B, n_groups, 1)
        # Weighted embeddings, then flatten
        weighted = (stacked * attn_weights).reshape(x.size(0), -1)  # (B, n_groups*embed_dim)
        return F.gelu(self.fuse(weighted))


print(f"PhysicsInformedWeatherEmbedding defined  "
      f"({len([g for g in FEATURE_GROUPS if len(FEATURE_GROUPS[g])>0])} active groups)")

PhysicsInformedWeatherEmbedding defined  (8 active groups)


### 3b. Model 1 — Baseline MLP (3-layer)

In [ ]:
class BaselineMLP(nn.Module):
    """Simple 3-hidden-layer MLP with dropout and batch normalisation."""
    def __init__(self, in_dim, out_dim, h1=256, h2=128, h3=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h1, h2),     nn.BatchNorm1d(h2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h2, h3),     nn.BatchNorm1d(h3), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(h3, out_dim),
        )
    def forward(self, x):
        return self.net(x)

print("Model 1: BaselineMLP defined.")

Model 1: BaselineMLP defined.


### 3c. Model 2 — MLP + Weather Embedding

In [ ]:
class WeatherMLP(nn.Module):
    """MLP that concatenates physics-informed weather embedding with raw features."""
    def __init__(self, in_dim, out_dim, feature_groups,
                 group_embed_dim=32, h1=256, h2=128, h3=64, dropout=0.3):
        super().__init__()
        self.weather_embed = PhysicsInformedWeatherEmbedding(
            feature_groups, group_embed_dim=group_embed_dim
        )
        combined_dim = in_dim + self.weather_embed.out_dim
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, h1), nn.BatchNorm1d(h1), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h1, h2),           nn.BatchNorm1d(h2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h2, h3),           nn.BatchNorm1d(h3), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(h3, out_dim),
        )
    
    def forward(self, x):
        w_emb = self.weather_embed(x)
        combined = torch.cat([x, w_emb], dim=-1)
        return self.classifier(combined)

print("Model 2: WeatherMLP defined.")

Model 2: WeatherMLP defined.


### 3d. Model 3 — Autoencoder + Classifier

In [ ]:
class AutoencoderClassifier(nn.Module):
    """Learns a compressed bottleneck via reconstruction, then classifies from it.
    
    Architecture:
        Encoder: in_dim → enc1 → enc2 → bottleneck_dim
        Decoder: bottleneck_dim → enc2 → enc1 → in_dim  (reconstruction)
        Classifier: bottleneck_dim → cls_h → out_dim
    
    Returns (logits, reconstruction) during training.
    """
    def __init__(self, in_dim, out_dim, enc1=256, enc2=128, bottleneck=64,
                 cls_hidden=64, dropout=0.3):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, enc1), nn.BatchNorm1d(enc1), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(enc1, enc2),   nn.BatchNorm1d(enc2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(enc2, bottleneck), nn.BatchNorm1d(bottleneck), nn.GELU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, enc2), nn.GELU(),
            nn.Linear(enc2, enc1),       nn.GELU(),
            nn.Linear(enc1, in_dim),
        )
        # Classifier head from bottleneck
        self.classifier = nn.Sequential(
            nn.Linear(bottleneck, cls_hidden), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(cls_hidden, out_dim),
        )
    
    def encode(self, x):
        return self.encoder(x)
    
    def forward(self, x):
        z = self.encode(x)
        recon = self.decoder(z)
        logits = self.classifier(z)
        return logits, recon

print("Model 3: AutoencoderClassifier defined.")

Model 3: AutoencoderClassifier defined.


### 3e. Model 4 — Autoencoder + Weather Embedding

In [ ]:
class WeatherAutoencoder(nn.Module):
    """Autoencoder whose bottleneck is enriched with the physics-informed weather embedding.
    
    Architecture:
        Weather Embed: features → group sub-nets → attention-fused vector
        Encoder: [raw_features | weather_embed] → enc1 → enc2 → bottleneck
        Decoder: bottleneck → dec2 → dec1 → in_dim  (reconstructs raw features only)
        Classifier: bottleneck → cls_h → out_dim
    """
    def __init__(self, in_dim, out_dim, feature_groups,
                 group_embed_dim=32, enc1=256, enc2=128, bottleneck=64,
                 cls_hidden=64, dropout=0.3):
        super().__init__()
        self.weather_embed = PhysicsInformedWeatherEmbedding(
            feature_groups, group_embed_dim=group_embed_dim
        )
        combined_in = in_dim + self.weather_embed.out_dim
        
        self.encoder = nn.Sequential(
            nn.Linear(combined_in, enc1), nn.BatchNorm1d(enc1), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(enc1, enc2),        nn.BatchNorm1d(enc2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(enc2, bottleneck),  nn.BatchNorm1d(bottleneck), nn.GELU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, enc2),  nn.GELU(),
            nn.Linear(enc2, enc1),        nn.GELU(),
            nn.Linear(enc1, in_dim),  # reconstruct raw features (no embedding)
        )
        self.classifier = nn.Sequential(
            nn.Linear(bottleneck, cls_hidden), nn.GELU(), nn.Dropout(dropout * 0.5),
            nn.Linear(cls_hidden, out_dim),
        )
    
    def forward(self, x):
        w_emb = self.weather_embed(x)
        combined = torch.cat([x, w_emb], dim=-1)
        z = self.encoder(combined)
        recon = self.decoder(z)  # reconstructs raw x, not the combined input
        logits = self.classifier(z)
        return logits, recon

print("Model 4: WeatherAutoencoder defined.")

Model 4: WeatherAutoencoder defined.


---
## 4. Hyperparameter Tuning with Optuna

Each model gets its own Optuna study. We use **Tree-structured Parzen Estimator (TPE)**
with 30 trials per model. The objective is to maximise **macro F1 on the validation set**.

In [ ]:
N_OPTUNA_TRIALS = 30
TUNING_EPOCHS   = 40           # shorter training during search
TUNING_PATIENCE = 8
FINAL_EPOCHS    = 80
FINAL_PATIENCE  = 15

# We'll collect all results here
all_results = {}   # model_name -> (overall_dict, per_label_df, history, thresholds)

print(f"Optuna config: {N_OPTUNA_TRIALS} trials, {TUNING_EPOCHS} epochs/trial")

Optuna config: 30 trials, 40 epochs/trial


### 4a. Tune & Train — Model 1: Baseline MLP

In [ ]:
def objective_mlp(trial):
    h1 = trial.suggest_categorical('h1', [128, 256, 512])
    h2 = trial.suggest_categorical('h2', [64, 128, 256])
    h3 = trial.suggest_categorical('h3', [32, 64, 128])
    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    bs = trial.suggest_categorical('batch_size', [512, 1024, 2048])
    
    train_loader, val_loader, _ = make_loaders(X_tr_np, y_tr_np, X_val_np, y_val_np,
                                               X_te_np, y_test_raw, batch_size=bs)
    model = BaselineMLP(n_features, n_labels, h1=h1, h2=h2, h3=h3, dropout=dropout)
    _, _, best_f1 = train_model(model, train_loader, val_loader, pos_weight_tensor,
                                lr=lr, weight_decay=wd, epochs=TUNING_EPOCHS,
                                patience=TUNING_PATIENCE, verbose=False)
    return best_f1

print("Tuning Model 1: Baseline MLP ...")
t0 = time.time()
study_mlp = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_mlp.optimize(objective_mlp, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print(f"  Done in {time.time()-t0:.0f}s")
print(f"  Best val F1: {study_mlp.best_value:.4f}")
print(f"  Best params: {study_mlp.best_params}")

Tuning Model 1: Baseline MLP ...


Best trial: 0. Best value: 0.156726:   3%|▎         | 1/30 [04:22<2:06:51, 262.48s/it]

[W 2026-02-20 05:18:11,883] Trial 1 failed with parameters: {'h1': 512, 'h2': 256, 'h3': 128, 'dropout': 0.30000000000000004, 'lr': 0.0021576967455896826, 'weight_decay': 3.972110727381911e-06, 'batch_size': 1024} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/preet/github_repos/cascade_env/lib/python3.9/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/dx/nxl1lszn07n8y3gzx2cg9ht00000gn/T/ipykernel_55031/3290522185.py", line 13, in objective_mlp
    _, _, best_f1 = train_model(model, train_loader, val_loader, pos_weight_tensor,
  File "/var/folders/dx/nxl1lszn07n8y3gzx2cg9ht00000gn/T/ipykernel_55031/2947761171.py", line 25, in train_model
    out = model(xb)
  File "/Users/preet/github_repos/cascade_env/lib/python3.9/site-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "/Users/preet/githu

KeyboardInterrupt: 

In [ ]:
# ── Retrain Model 1 with best params for full epochs ──────────────────
bp = study_mlp.best_params
train_loader, val_loader, test_loader = make_loaders(
    X_tr_np, y_tr_np, X_val_np, y_val_np, X_te_np, y_test_raw,
    batch_size=bp['batch_size']
)

print("\nTraining Model 1 (final) ...")
model1 = BaselineMLP(n_features, n_labels, h1=bp['h1'], h2=bp['h2'],
                     h3=bp['h3'], dropout=bp['dropout'])
model1, hist1, _ = train_model(model1, train_loader, val_loader, pos_weight_tensor,
                               lr=bp['lr'], weight_decay=bp['weight_decay'],
                               epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE)

# Predict & evaluate
y_prob_val1 = predict_proba(model1, val_loader)
thresh1 = tune_thresholds(y_val_np, y_prob_val1)
y_prob_test1 = predict_proba(model1, test_loader)
overall1, perlabel1 = evaluate_model('1_BaselineMLP', y_test_raw, y_prob_test1, thresh1, target_names)
all_results['1_BaselineMLP'] = (overall1, perlabel1, hist1, thresh1)

print(f"\n=== Model 1 Test Results ===")
for k, v in overall1.items():
    if k != 'model':
        print(f"  {k:25s} {v:.4f}")


Training Model 1 (final) ...
  Epoch   1  loss=0.3668  val_loss=0.2021  val_f1=0.0868 ★
  Epoch   2  loss=0.1725  val_loss=0.1481  val_f1=0.1153 ★
  Epoch   3  loss=0.1478  val_loss=0.1346  val_f1=0.1233 ★
  Epoch   4  loss=0.1395  val_loss=0.1296  val_f1=0.1274 ★


KeyboardInterrupt: 

### 4b. Tune & Train — Model 2: MLP + Weather Embedding

In [ ]:
def objective_weather_mlp(trial):
    group_embed_dim = trial.suggest_categorical('group_embed_dim', [16, 32, 48])
    h1 = trial.suggest_categorical('h1', [128, 256, 512])
    h2 = trial.suggest_categorical('h2', [64, 128, 256])
    h3 = trial.suggest_categorical('h3', [32, 64, 128])
    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    bs = trial.suggest_categorical('batch_size', [512, 1024, 2048])
    
    train_loader, val_loader, _ = make_loaders(X_tr_np, y_tr_np, X_val_np, y_val_np,
                                               X_te_np, y_test_raw, batch_size=bs)
    model = WeatherMLP(n_features, n_labels, FEATURE_GROUPS,
                       group_embed_dim=group_embed_dim, h1=h1, h2=h2, h3=h3, dropout=dropout)
    _, _, best_f1 = train_model(model, train_loader, val_loader, pos_weight_tensor,
                                lr=lr, weight_decay=wd, epochs=TUNING_EPOCHS,
                                patience=TUNING_PATIENCE, verbose=False)
    return best_f1

print("Tuning Model 2: MLP + Weather Embedding ...")
t0 = time.time()
study_weather_mlp = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_weather_mlp.optimize(objective_weather_mlp, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print(f"  Done in {time.time()-t0:.0f}s")
print(f"  Best val F1: {study_weather_mlp.best_value:.4f}")
print(f"  Best params: {study_weather_mlp.best_params}")

Tuning Model 2: MLP + Weather Embedding ...


  0%|          | 0/30 [00:00<?, ?it/s]

[W 2026-02-20 05:12:20,709] Trial 0 failed with parameters: {'group_embed_dim': 32, 'h1': 128, 'h2': 128, 'h3': 128, 'dropout': 0.45000000000000007, 'lr': 0.00022948683681130568, 'weight_decay': 3.5113563139704077e-06, 'batch_size': 2048} because of the following error: TypeError("__init__() got an unexpected keyword argument 'verbose'").
Traceback (most recent call last):
  File "/Users/preet/github_repos/cascade_env/lib/python3.9/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/dx/nxl1lszn07n8y3gzx2cg9ht00000gn/T/ipykernel_55031/1871853681.py", line 15, in objective_weather_mlp
    _, _, best_f1 = train_model(model, train_loader, val_loader, pos_weight_tensor,
  File "/var/folders/dx/nxl1lszn07n8y3gzx2cg9ht00000gn/T/ipykernel_55031/4166511133.py", line 8, in train_model
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=False)
TypeError: __init__() got an unexpected keyword a

TypeError: __init__() got an unexpected keyword argument 'verbose'

In [ ]:
# ── Retrain Model 2 with best params ──────────────────────────────────
bp2 = study_weather_mlp.best_params
train_loader, val_loader, test_loader = make_loaders(
    X_tr_np, y_tr_np, X_val_np, y_val_np, X_te_np, y_test_raw,
    batch_size=bp2['batch_size']
)

print("\nTraining Model 2 (final) ...")
model2 = WeatherMLP(n_features, n_labels, FEATURE_GROUPS,
                    group_embed_dim=bp2['group_embed_dim'],
                    h1=bp2['h1'], h2=bp2['h2'], h3=bp2['h3'], dropout=bp2['dropout'])
model2, hist2, _ = train_model(model2, train_loader, val_loader, pos_weight_tensor,
                               lr=bp2['lr'], weight_decay=bp2['weight_decay'],
                               epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE)

y_prob_val2 = predict_proba(model2, val_loader)
thresh2 = tune_thresholds(y_val_np, y_prob_val2)
y_prob_test2 = predict_proba(model2, test_loader)
overall2, perlabel2 = evaluate_model('2_WeatherMLP', y_test_raw, y_prob_test2, thresh2, target_names)
all_results['2_WeatherMLP'] = (overall2, perlabel2, hist2, thresh2)

print(f"\n=== Model 2 Test Results ===")
for k, v in overall2.items():
    if k != 'model':
        print(f"  {k:25s} {v:.4f}")

### 4c. Tune & Train — Model 3: Autoencoder + Classifier

In [ ]:
def objective_ae(trial):
    enc1 = trial.suggest_categorical('enc1', [128, 256, 512])
    enc2 = trial.suggest_categorical('enc2', [64, 128, 256])
    bottleneck = trial.suggest_categorical('bottleneck', [32, 48, 64, 96])
    cls_hidden = trial.suggest_categorical('cls_hidden', [32, 64, 128])
    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    recon_w = trial.suggest_float('recon_weight', 0.01, 1.0, log=True)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    bs = trial.suggest_categorical('batch_size', [512, 1024, 2048])
    
    train_loader, val_loader, _ = make_loaders(X_tr_np, y_tr_np, X_val_np, y_val_np,
                                               X_te_np, y_test_raw, batch_size=bs)
    model = AutoencoderClassifier(n_features, n_labels, enc1=enc1, enc2=enc2,
                                  bottleneck=bottleneck, cls_hidden=cls_hidden, dropout=dropout)
    _, _, best_f1 = train_model(model, train_loader, val_loader, pos_weight_tensor,
                                lr=lr, weight_decay=wd, epochs=TUNING_EPOCHS,
                                patience=TUNING_PATIENCE, verbose=False,
                                recon_weight=recon_w)
    return best_f1

print("Tuning Model 3: Autoencoder + Classifier ...")
t0 = time.time()
study_ae = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_ae.optimize(objective_ae, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print(f"  Done in {time.time()-t0:.0f}s")
print(f"  Best val F1: {study_ae.best_value:.4f}")
print(f"  Best params: {study_ae.best_params}")

In [ ]:
# ── Retrain Model 3 with best params ──────────────────────────────────
bp3 = study_ae.best_params
train_loader, val_loader, test_loader = make_loaders(
    X_tr_np, y_tr_np, X_val_np, y_val_np, X_te_np, y_test_raw,
    batch_size=bp3['batch_size']
)

print("\nTraining Model 3 (final) ...")
model3 = AutoencoderClassifier(n_features, n_labels, enc1=bp3['enc1'], enc2=bp3['enc2'],
                               bottleneck=bp3['bottleneck'], cls_hidden=bp3['cls_hidden'],
                               dropout=bp3['dropout'])
model3, hist3, _ = train_model(model3, train_loader, val_loader, pos_weight_tensor,
                               lr=bp3['lr'], weight_decay=bp3['weight_decay'],
                               epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE,
                               recon_weight=bp3['recon_weight'])

y_prob_val3 = predict_proba(model3, val_loader)
thresh3 = tune_thresholds(y_val_np, y_prob_val3)
y_prob_test3 = predict_proba(model3, test_loader)
overall3, perlabel3 = evaluate_model('3_Autoencoder', y_test_raw, y_prob_test3, thresh3, target_names)
all_results['3_Autoencoder'] = (overall3, perlabel3, hist3, thresh3)

print(f"\n=== Model 3 Test Results ===")
for k, v in overall3.items():
    if k != 'model':
        print(f"  {k:25s} {v:.4f}")

### 4d. Tune & Train — Model 4: Autoencoder + Weather Embedding

In [ ]:
def objective_weather_ae(trial):
    group_embed_dim = trial.suggest_categorical('group_embed_dim', [16, 32, 48])
    enc1 = trial.suggest_categorical('enc1', [128, 256, 512])
    enc2 = trial.suggest_categorical('enc2', [64, 128, 256])
    bottleneck = trial.suggest_categorical('bottleneck', [32, 48, 64, 96])
    cls_hidden = trial.suggest_categorical('cls_hidden', [32, 64, 128])
    dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.05)
    recon_w = trial.suggest_float('recon_weight', 0.01, 1.0, log=True)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    bs = trial.suggest_categorical('batch_size', [512, 1024, 2048])
    
    train_loader, val_loader, _ = make_loaders(X_tr_np, y_tr_np, X_val_np, y_val_np,
                                               X_te_np, y_test_raw, batch_size=bs)
    model = WeatherAutoencoder(n_features, n_labels, FEATURE_GROUPS,
                               group_embed_dim=group_embed_dim, enc1=enc1, enc2=enc2,
                               bottleneck=bottleneck, cls_hidden=cls_hidden, dropout=dropout)
    _, _, best_f1 = train_model(model, train_loader, val_loader, pos_weight_tensor,
                                lr=lr, weight_decay=wd, epochs=TUNING_EPOCHS,
                                patience=TUNING_PATIENCE, verbose=False,
                                recon_weight=recon_w)
    return best_f1

print("Tuning Model 4: Weather Autoencoder ...")
t0 = time.time()
study_weather_ae = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study_weather_ae.optimize(objective_weather_ae, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
print(f"  Done in {time.time()-t0:.0f}s")
print(f"  Best val F1: {study_weather_ae.best_value:.4f}")
print(f"  Best params: {study_weather_ae.best_params}")

In [ ]:
# ── Retrain Model 4 with best params ──────────────────────────────────
bp4 = study_weather_ae.best_params
train_loader, val_loader, test_loader = make_loaders(
    X_tr_np, y_tr_np, X_val_np, y_val_np, X_te_np, y_test_raw,
    batch_size=bp4['batch_size']
)

print("\nTraining Model 4 (final) ...")
model4 = WeatherAutoencoder(n_features, n_labels, FEATURE_GROUPS,
                            group_embed_dim=bp4['group_embed_dim'],
                            enc1=bp4['enc1'], enc2=bp4['enc2'],
                            bottleneck=bp4['bottleneck'], cls_hidden=bp4['cls_hidden'],
                            dropout=bp4['dropout'])
model4, hist4, _ = train_model(model4, train_loader, val_loader, pos_weight_tensor,
                               lr=bp4['lr'], weight_decay=bp4['weight_decay'],
                               epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE,
                               recon_weight=bp4['recon_weight'])

y_prob_val4 = predict_proba(model4, val_loader)
thresh4 = tune_thresholds(y_val_np, y_prob_val4)
y_prob_test4 = predict_proba(model4, test_loader)
overall4, perlabel4 = evaluate_model('4_WeatherAE', y_test_raw, y_prob_test4, thresh4, target_names)
all_results['4_WeatherAE'] = (overall4, perlabel4, hist4, thresh4)

print(f"\n=== Model 4 Test Results ===")
for k, v in overall4.items():
    if k != 'model':
        print(f"  {k:25s} {v:.4f}")

---
## 5. Comparative Analysis

### 5a. Overall Metrics Comparison

In [ ]:
# ── Side-by-side overall metrics ──────────────────────────────────────
comparison_df = pd.DataFrame([v[0] for v in all_results.values()])
comparison_df = comparison_df.set_index('model')

print("\n" + "="*80)
print("                    OVERALL METRICS COMPARISON")
print("="*80)
display(comparison_df.style.format('{:.4f}').highlight_max(axis=0, color='#90ee90')
        .highlight_min(axis=0, color='#ffcccb'))

# Bar chart
metrics_to_plot = ['f1_weighted', 'f1_macro', 'f1_micro', 'precision_macro', 'recall_macro']
plot_df = comparison_df[metrics_to_plot].T

fig, ax = plt.subplots(figsize=(14, 6))
plot_df.plot(kind='bar', ax=ax, edgecolor='white', width=0.8)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Overall Metrics')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_ylim(0, comparison_df[metrics_to_plot].values.max() * 1.15)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7, padding=2)
plt.tight_layout()
plt.show()

### 5b. Per-Label F1 Comparison

In [ ]:
# ── Per-label F1 heatmap across all models ────────────────────────────
perlabel_dict = {}
for mname, (_, pldf, _, _) in all_results.items():
    perlabel_dict[mname] = pldf.set_index('label')['f1']

f1_comparison = pd.DataFrame(perlabel_dict)
f1_comparison['best_model'] = f1_comparison.idxmax(axis=1)
f1_comparison = f1_comparison.sort_values(f1_comparison.columns[0], ascending=False)

fig, ax = plt.subplots(figsize=(12, max(8, len(target_names) * 0.45)))
sns.heatmap(f1_comparison.drop(columns=['best_model']).astype(float),
            annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, linewidths=.5,
            cbar_kws={'label': 'F1 Score'})
ax.set_title('Per-Label F1: All Models', fontsize=14)
ax.set_xlabel('Model')
plt.tight_layout()
plt.show()

# Who wins per label?
print("\nBest model per label:")
for label, row in f1_comparison.iterrows():
    print(f"  {label:35s}  →  {row['best_model']}")

### 5c. Training Curves

In [ ]:
# ── Training curves: loss & val F1 ────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, (mname, (_, _, hist, _)) in zip(axes.flat, all_results.items()):
    epochs_range = range(1, len(hist['train_loss']) + 1)
    
    ax2 = ax.twinx()
    l1, = ax.plot(epochs_range, hist['train_loss'], 'b-', alpha=0.6, label='Train Loss')
    l2, = ax.plot(epochs_range, hist['val_loss'],   'r-', alpha=0.6, label='Val Loss')
    l3, = ax2.plot(epochs_range, hist['val_f1'],    'g--', alpha=0.8, label='Val F1 (macro)')
    
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss', color='blue')
    ax2.set_ylabel('F1', color='green')
    ax.set_title(mname.replace('_', ' '))
    ax.legend(handles=[l1, l2, l3], fontsize=8, loc='upper right')

fig.suptitle('Training Curves — All Models', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5d. Optuna Tuning Visualisation

In [ ]:
# ── Optuna trial history ──────────────────────────────────────────────
studies = {
    '1_BaselineMLP': study_mlp,
    '2_WeatherMLP':  study_weather_mlp,
    '3_Autoencoder': study_ae,
    '4_WeatherAE':   study_weather_ae,
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (name, study) in zip(axes.flat, studies.items()):
    trials = study.trials
    values = [t.value for t in trials if t.value is not None]
    ax.plot(range(1, len(values)+1), values, 'o-', markersize=4, alpha=0.7)
    best_so_far = np.maximum.accumulate(values)
    ax.plot(range(1, len(values)+1), best_so_far, 'r-', lw=2, label='Best so far')
    ax.set_xlabel('Trial')
    ax.set_ylabel('Val F1 (macro)')
    ax.set_title(name.replace('_', ' '))
    ax.legend(fontsize=8)

fig.suptitle('Optuna Hyperparameter Search Progress', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5e. Precision-Recall Curves (Best Model)

In [ ]:
# ── PR curves for the best-performing model ───────────────────────────
best_model_name = comparison_df['f1_macro'].idxmax()
_, best_perlabel, _, _ = all_results[best_model_name]
# Get the matching y_prob
best_probs = {'1_BaselineMLP': y_prob_test1, '2_WeatherMLP': y_prob_test2,
              '3_Autoencoder': y_prob_test3, '4_WeatherAE': y_prob_test4}[best_model_name]

show_pr_labels = best_perlabel.nlargest(8, 'pr_auc')['label'].tolist()

fig, ax = plt.subplots(figsize=(10, 8))
for label in show_pr_labels:
    idx = target_names.index(label)
    if y_test_raw[:, idx].sum() == 0:
        continue
    precs, recs, _ = precision_recall_curve(y_test_raw[:, idx], best_probs[:, idx])
    ap = average_precision_score(y_test_raw[:, idx], best_probs[:, idx])
    ax.plot(recs, precs, label=f'{label} (AP={ap:.3f})', lw=1.5)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title(f'PR Curves — {best_model_name} (top 8 labels by PR-AUC)')
ax.legend(loc='upper right', fontsize=8)
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)
plt.tight_layout()
plt.show()

### 5f. Confusion Matrices (Best Model, Top 6 Labels)

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────
_, best_pl, _, best_thresh = all_results[best_model_name]
y_pred_best = (best_probs >= best_thresh).astype(int)
cm_labels = best_pl.nlargest(6, 'support')['label'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, label in zip(axes.flat, cm_labels):
    idx = target_names.index(label)
    cm = confusion_matrix(y_test_raw[:, idx], y_pred_best[:, idx])
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    ax.set_title(label)
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

fig.suptitle(f'Confusion Matrices — {best_model_name}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 5g. Autoencoder Bottleneck Visualisation

In [ ]:
# ── Visualise the autoencoder bottleneck (Model 3 & 4) ────────────────
from sklearn.decomposition import PCA

SAMPLE_N = min(3000, X_te_np.shape[0])
rng = np.random.RandomState(42)
sample_idx = rng.choice(X_te_np.shape[0], SAMPLE_N, replace=False)
X_sample = torch.tensor(X_te_np[sample_idx], dtype=torch.float32).to(DEVICE)
y_sample = y_test_raw[sample_idx]

# Get bottleneck embeddings
model3.eval()
model4.eval()
with torch.no_grad():
    z3 = model3.encode(X_sample).cpu().numpy()
    # For model4 we need to pass through weather embed + encoder
    w_emb4 = model4.weather_embed(X_sample)
    combined4 = torch.cat([X_sample, w_emb4], dim=-1)
    z4 = model4.encoder(combined4).cpu().numpy()

# PCA to 2D
pca3 = PCA(n_components=2, random_state=42)
coords3 = pca3.fit_transform(z3)
pca4 = PCA(n_components=2, random_state=42)
coords4 = pca4.fit_transform(z4)

# Colour by total cascade count
label_count = y_sample.sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, coords, pca_obj, title in [
    (axes[0], coords3, pca3, 'Model 3: Autoencoder'),
    (axes[1], coords4, pca4, 'Model 4: Weather Autoencoder'),
]:
    sc = ax.scatter(coords[:, 0], coords[:, 1], c=label_count, cmap='viridis',
                    alpha=0.5, s=8, edgecolors='none')
    ax.set_xlabel(f'PC1 ({pca_obj.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca_obj.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(title)
    plt.colorbar(sc, ax=ax, label='# Active Labels')

fig.suptitle('Autoencoder Bottleneck Embeddings (PCA)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5h. Weather Embedding Attention Weights

In [ ]:
# ── Inspect attention weights in the weather embedding ─────────────────
def get_attention_weights(model, X_sample):
    """Extract attention weights from PhysicsInformedWeatherEmbedding."""
    we = model.weather_embed if hasattr(model, 'weather_embed') else None
    if we is None:
        return None, None
    
    we.eval()
    group_embeds = []
    group_names_active = []
    with torch.no_grad():
        for gname in we.group_names:
            if gname not in we.sub_networks:
                continue
            idx = we.feature_groups[gname]
            x_group = X_sample[:, idx]
            group_embeds.append(we.sub_networks[gname](x_group))
            group_names_active.append(gname)
        
        stacked = torch.stack(group_embeds, dim=1)
        attn_scores = we.attn_query(stacked).squeeze(-1)
        attn_weights = F.softmax(attn_scores, dim=-1)  # (B, n_groups)
    
    return attn_weights.cpu().numpy(), group_names_active

# Get weights from both weather models
attn2, groups2 = get_attention_weights(model2, X_sample)
attn4, groups4 = get_attention_weights(model4, X_sample)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, attn, groups, title in [
    (axes[0], attn2, groups2, 'Model 2: WeatherMLP'),
    (axes[1], attn4, groups4, 'Model 4: WeatherAE'),
]:
    if attn is None:
        ax.text(0.5, 0.5, 'No weather embedding', ha='center', va='center')
        continue
    mean_attn = attn.mean(axis=0)
    std_attn  = attn.std(axis=0)
    sorted_idx = np.argsort(mean_attn)
    
    ax.barh([groups[i] for i in sorted_idx], mean_attn[sorted_idx],
            xerr=std_attn[sorted_idx], color='coral', edgecolor='white', capsize=3)
    ax.set_xlabel('Mean Attention Weight')
    ax.set_title(title)
    for i, idx in enumerate(sorted_idx):
        ax.text(mean_attn[idx] + std_attn[idx] + 0.005, i,
                f'{mean_attn[idx]:.3f}', va='center', fontsize=9)

fig.suptitle('Weather Embedding — Learned Attention over Feature Groups', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\nInterpretation: Higher attention weight = the model finds this feature group")
print("more useful for cascade prediction after physics-informed encoding.")

---
## 6. Final Summary & Conclusion

In [ ]:
# ── Final comparison table ─────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════════════════╗')
print('║               NEURAL NETWORK MODEL COMPARISON SUMMARY                  ║')
print('╠══════════════════════════════════════════════════════════════════════════╣')
print(f'║  {"Model":25s}  {"F1-w":>6s}  {"F1-macro":>8s}  {"F1-micro":>8s}  '
      f'{"Prec-M":>7s}  {"Rec-M":>6s}  {"HL":>6s}  ║')
print('║' + '-' * 74 + '║')
for mname, (res, _, _, _) in all_results.items():
    print(f'║  {mname:25s}  {res["f1_weighted"]:6.4f}  {res["f1_macro"]:8.4f}  '
          f'{res["f1_micro"]:8.4f}  {res["precision_macro"]:7.4f}  '
          f'{res["recall_macro"]:6.4f}  {res["hamming_loss"]:6.4f}  ║')
print('╚══════════════════════════════════════════════════════════════════════════╝')

# Determine best
best_name = max(all_results.keys(), key=lambda k: all_results[k][0]['f1_macro'])
print(f'\n  Best model by macro-F1: {best_name}')
print(f'  XGBoost baseline (from notebook 02): F1-weighted ≈ 0.4697, F1-macro ≈ 0.2963')

# Model parameter counts
print('\n  Parameter counts:')
for name, mdl in [('1_BaselineMLP', model1), ('2_WeatherMLP', model2),
                   ('3_Autoencoder', model3), ('4_WeatherAE', model4)]:
    n_params = sum(p.numel() for p in mdl.parameters())
    print(f'    {name:25s}  {n_params:>10,} params')

In [ ]:
# ── Save all models ────────────────────────────────────────────────────
SAVE_DIR = Path('../models/neural_nets')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

for name, mdl in [('model1_baseline_mlp', model1), ('model2_weather_mlp', model2),
                   ('model3_autoencoder', model3), ('model4_weather_ae', model4)]:
    torch.save(mdl.state_dict(), SAVE_DIR / f'{name}.pt')

# Save results
with open(SAVE_DIR / 'results_summary.pkl', 'wb') as f:
    pickle.dump({
        'all_results': {k: (v[0], v[1], v[3].tolist()) for k, v in all_results.items()},
        'comparison_df': comparison_df,
        'best_params': {
            '1_BaselineMLP': study_mlp.best_params,
            '2_WeatherMLP':  study_weather_mlp.best_params,
            '3_Autoencoder': study_ae.best_params,
            '4_WeatherAE':   study_weather_ae.best_params,
        }
    }, f)

print(f"Models & results saved to {SAVE_DIR}/")

## Conclusion

This notebook implemented and compared four neural network architectures for multilabel cascading disaster prediction. All models were tuned via Optuna (30 trials each) and evaluated using the same chronological train/val/test split and per-label threshold tuning as the XGBoost baseline.

**Key architectural innovations:**

- **Physics-Informed Weather Embedding**: A trainable module that separates features into meteorological groups (temporal-cyclical, spatial, severity, event-type, cascade-transition probabilities, historical context) and applies group-specific sub-networks: cyclical gating for periodic time features, RBF spatial transforms for geographic coordinates, and softmax normalization for cascade probability vectors. Group representations are fused via learned attention weights, giving the model an inductive bias toward physically meaningful feature interactions.

- **Autoencoder Bottleneck**: Forces the model to learn a compressed representation that captures the most salient patterns in the input data. The reconstruction loss acts as a regularizer, preventing the classifier from over-fitting to label noise.

**What to look for in the results:**
- Do weather embeddings improve over raw-feature MLPs? (Models 2 & 4 vs 1 & 3)
- Does the autoencoder's bottleneck help or hurt? (Models 3 & 4 vs 1 & 2)
- Which feature groups receive the highest attention weights?
- How do the neural net models compare to the XGBoost baseline (F1-weighted ≈ 0.47, F1-macro ≈ 0.30)?